# 第九部分：Agent 的诞生 —— 解码、微调与结构化约束

> **本章目标**：将预训练模型改造为听你话、不乱说、会调API的生产级Agent。

## 第33章　解码策略的"人格分裂"实验 —— 从贪心到核采样

> 本章目标：不只讲采样公式，而是带着读者在"输出质量"和"多样性"之间做工程调参实验。
> 前置知识：第 15 章（采样策略）、第 22 章（softmax/Logit Bias）、第 32 章（KV Cache 推理）

> 🕵️ **开篇导语：审讯策略的"人格分裂"实验**
>
> 设备推理飞快，但它面临一个更微妙的问题：**每次只输出一个 Token，该选哪个？**
>
> 如果永远选概率最高的那个（**Greedy 策略**），你会发现它像一位固执的老刑警，翻来覆去只说同一句话——"我怀疑是你，就是你，肯定是你"。
>
> **Top-p（核采样）** 是更优雅的方案：从概率最高的 Token 开始累加，直到累计概率达到 `p=0.9`。候选集随上下文动态变化。这就是当前所有 SOTA Agent 的默认"性格开关"。
>
> 本章没有"正确答案"——只有"合适的选择"。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("env ready")


### 33.1　Greedy 的"复读机"诅咒

💻 **代码　Greedy 生成 50 个 Token，观察重复模式**


In [ ]:
import numpy as np

np.random.seed(42)
vocab_size, seq_len = 100, 50
logits = np.random.randn(seq_len, vocab_size) * 0.5
logits[:, 0] += 2.0  # token 0 始终最高概率

tokens = np.argmax(logits, axis=1)
repeats = sum(1 for i in range(1, len(tokens)) if tokens[i] == tokens[i-1])
unique_ratio = len(np.unique(tokens)) / len(tokens)

print(f"Greedy tokens: {tokens}")
print(f"Unique ratio: {unique_ratio:.0%}, Adjacent repeats: {repeats}/{seq_len-1}")


---


### 33.2　Temperature 的混沌效应

📐 **定义　Temperature**：`p_i = exp(z_i/T) / Σ exp(z_j/T)`。T→0 退化为 Greedy，T→∞ 趋近均匀分布。

💻 **代码　Temperature 对概率分布的影响**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

logits = np.array([3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1])

def softmax(x, T=1.0):
    x = np.float64(x)/T; x = x-x.max(); e = np.exp(x); return e/e.sum()

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, T in zip(axes, [0.1, 0.5, 1.0, 2.0]):
    probs = softmax(logits, T)
    ax.bar(range(len(logits)), probs, color='steelblue', edgecolor='white')
    ax.set_title(f'T={T}')
    ax.set_xlabel('Token'); ax.set_ylabel('Prob')
plt.suptitle('Temperature: Control the Entropy of Softmax', fontsize=13)
plt.tight_layout(); plt.show()


---


### 33.3　Top-k 的"截断暴力"

📐 **定义　Top-k**：保留概率最高的 k 个 token，置零其余，重归一化。k=50 常见默认。固定 k 不能自适应。

---


### 33.4　Top-p（核采样）的动态哲学

📐 **定义　Top-p**：从 argmax 累加概率，直到 Σp ≥ p。候选集动态大小。p=0.9 工业标准。

💻 **代码　Top-k vs Top-p：候选集大小对比**


In [ ]:
import numpy as np

def softmax(x): x=np.float64(x); x=x-x.max(); e=np.exp(x); return e/e.sum()

logits_c = np.array([5.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
logits_f = np.array([2.0, 1.9, 1.8, 1.7, 1.6, 1.5, 1.4, 1.3])

for name, logits in [("集中", logits_c), ("分散", logits_f)]:
    probs = softmax(logits)
    si = np.argsort(probs)[::-1]; cs = np.cumsum(probs[si])
    k3_cov = probs[si[:3]].sum()
    n_p = np.searchsorted(cs, 0.9) + 1
    print(f"{name}: Top-k=3 covers {k3_cov:.0%}, Top-p=0.9 keeps {n_p} tokens")


---


### 33.5　惩罚机制（Repetition Penalty）

📐 **定义　Repetition Penalty**：对已出现 token 的 logit 减去 penalty，软约束防止重复。

💻 **代码　实现 repetition_penalty**


In [ ]:
import numpy as np

def softmax(x): x=np.float64(x); x=x-x.max(); e=np.exp(x); return e/e.sum()

logits = np.ones(10) * 2.0
generated = [0, 3, 0]
penalty = 1.5

lp = logits.copy()
for tid in set(generated): lp[tid] -= penalty

po = softmax(logits); pp = softmax(lp)
print(f"Token 0 prob: {po[0]:.1%} -> {pp[0]:.1%} (penalized)")


---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. 对同一个 Prompt，用四种策略各生成 100 段文本，计算 Distinct-4 指标并排序。
2. 实现 `repetition_penalty`，在 `"I love"` 开头下验证重复被抑制。
3. 画出 `Temperature × Distinct-4` 曲线图，找帕累托最优区间。
---
> 🔗 **章末钩子**：模型生成更像人话了。但如果要让它学会用特定格式调用工具，该怎么"教"它？→ 引向第 34 章。
> 预览：下一章——**微调落地的"手术刀"**，从 PyTorch 基础到 LoRA。
